# Build the time series database

This notebook pulls raw data onto your machine and loads it into a local SQLite database (`timeseries.db`).  
Run the cells top to bottom. Steps 2 and 3 hit the internet — step 3 takes ~10 minutes.

| Step | What it does | Time |
|------|-------------|------|
| 0 | Install required Python packages | ~10 sec |
| 1 | Reads the OBR Excel spreadsheets already in `data/2026-03/obr/` | ~5 sec |
| 2 | Fetches ~30 core ONS economic series from the ONS API | ~30 sec |
| 3 | Fetches ONS mirrors for every OBR variable (optional but recommended) | ~10 min |

## Step 0 — Install dependencies

Only needs to run once. **After this cell finishes, restart the kernel** (Kernel → Restart), then run from the next cell down.

In [1]:
%pip install openpyxl requests pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Setup — paths and checks

In [ ]:
import os, sys

REPO = os.path.expanduser('~/Documents/CBP/budget-master')
sys.path.insert(0, REPO)

VINTAGE      = '2026-03'
OBR_DATA_DIR = os.path.join(REPO, 'data', VINTAGE, 'obr')
ONS_VARS_XLS = os.path.join(REPO, 'docs', 'OBR_Model_Variables_March_2025.xlsx')
DB_PATH      = os.path.join(REPO, 'shaamini_tests', 'timeseries.db')

assert os.path.isdir(OBR_DATA_DIR), f'OBR data folder not found: {OBR_DATA_DIR}'
assert os.path.isfile(ONS_VARS_XLS), f'ONS variable list not found: {ONS_VARS_XLS}'

print('OBR data folder:', OBR_DATA_DIR)
print('OBR files found:', len(os.listdir(OBR_DATA_DIR)))
print('Database path:  ', DB_PATH)

## Step 1 — Load OBR spreadsheets into the database

Reads the Excel files from `data/2026-03/obr/` and writes them into `timeseries.db`.  
Purely local — no internet needed.

In [3]:
from cbp_fiscal_framework.db.timeseries_db import TimeSeriesDB

db = TimeSeriesDB(DB_PATH)
db.build_from_obr(OBR_DATA_DIR, VINTAGE)

print('Step 1 done.')

Step 1 done.


## Step 2 — Fetch core ONS series (~30 seconds)

Downloads ~30 key economic series (GDP, CPI, employment, etc.) from the ONS API.

In [4]:
db.build_from_ons(ONS_VARS_XLS)

print('Step 2 done.')

/Users/shaaminiprinja/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Step 2 done.


## Step 3 — Fetch ONS mirrors (~10 minutes, optional)

Fetches the ONS equivalent for every OBR variable so they can be compared side by side.  
Skip this if you're in a hurry — Steps 1 and 2 are enough to run the model.

In [5]:
db.build_ons_mirrors()

print('Step 3 done.')

Step 3 done.


## Step 4 — Close and verify

In [6]:
db.close()

size_mb = os.path.getsize(DB_PATH) / 1024 / 1024
print(f'Database written to: {DB_PATH}')
print(f'Size: {size_mb:.1f} MB')

Database written to: /Users/shaaminiprinja/Downloads/budget-master/cbp_fiscal_framework/db/timeseries.db
Size: 2.2 MB


In [7]:
import sqlite3

con = sqlite3.connect(DB_PATH)

n_series = con.execute('SELECT COUNT(*) FROM series').fetchone()[0]
n_obs    = con.execute('SELECT COUNT(*) FROM observations').fetchone()[0]
sources  = [r[0] for r in con.execute('SELECT DISTINCT source FROM observations').fetchall()]

print(f'Series:       {n_series}')
print(f'Observations: {n_obs}')
print(f'Sources:      {sources}')

print('\nSample series:')
for row in con.execute('SELECT id, label, has_obr, has_ons FROM series LIMIT 10').fetchall():
    print(f'  {row[0]:<12} {str(row[1])[:40]:<42} OBR={row[2]} ONS={row[3]}')

con.close()

Series:       104
Observations: 15770
Sources:      ['OBR_EFO', 'ONS']

Sample series:
  CONS         HH (&NPISH) final consumption expenditur   OBR=1 ONS=0
  CGG          General Govt final consumption CVM         OBR=1 ONS=1
  IF           Total gross fixed capital formation (CVM   OBR=1 ONS=1
  IBUS         Business investment                        OBR=1 ONS=1
  IH           Private sector investment in dwellings     OBR=1 ONS=0
  GGI          General Government GFCF (CVM)              OBR=1 ONS=0
  DINV         Change in inventories (CVM)                OBR=1 ONS=1
  VAL          Net acquisitions of valuables (CVM)        OBR=1 ONS=1
  X            Total exports, CVM                         OBR=1 ONS=0
  M            Total imports (CVM)                        OBR=1 ONS=0
